# 01 · Quantize MANGO1.5-Qwen3.5-9B สำหรับ GreenMind Hackathon

> **ฉบับปรับให้รันบน VM แข่งขัน (A100-SXM4 40GB · RAM 47 GiB · Ubuntu 24.04) แทน Colab**
> - Kernel: เลือก **`GreenMind quant (llmcompressor 0.12)`** (venv `~/nut/venvs/quant`: llmcompressor 0.12.0, transformers 5.10.1, torch 2.12+cu132) ไม่ต้องรัน 0.1
> - ไม่มี Google Drive: toolkit แตกไปที่ `~/nut/gm`, ผลลัพธ์อยู่ที่ `~/nut/results`, checkpoint อยู่ที่ `~/nut/models`
> - ใช้ HF cache เดิม `~/work/hf` (มี MANGO BF16 18 GB แล้ว ไม่ต้องโหลดซ้ำ) และ vLLM venv เดิม `~/work/.venv` สำหรับ 0.7a
> - **ก่อนรัน:** GPU และ RAM ต้องว่าง (ห้ามมี kernel/vLLM ตัวอื่นถือ GPU หรือกิน RAM) cell 0.3 จะตรวจให้และหยุดถ้าไม่พร้อม
> - **ห้ามรันระหว่างส่ง Validation/Final** กติกาห้ามมี GPU workload อื่นระหว่าง graded run
> - สูตร 3–5 ถูก comment ไว้ เพราะ disk เหลือ ~45 GB (เก็บ checkpoint ได้ราว 2–3 ชุด) และเวลาก่อนปิด grader

**หน้าที่ของ notebook นี้ = หน้าที่ของคนทำ quantize** สร้าง checkpoint ที่เร็วขึ้นและใช้พลังงานน้อยลงบน A100, พิสูจน์ว่าคุณภาพภาษาไทยยังผ่าน, เก็บผลทุกรอบไว้บน Drive และสร้างชุดส่งมอบ (`HANDOFF.md`) ให้คนทำ serving

**วิธีใช้:** รันจากบนลงล่างทีละ cell และอ่าน "✅ ควรเห็น" ก่อนกด cell ถัดไป **ไม่ต้องแก้ตัวแปรใด ๆ** เพราะแต่ละสูตรมีส่วนของตัวเอง ส่วนละ 3 cell:
1. `lab.quantize(...)` โหลดโมเดล BF16 ใหม่ → ใส่ recipe → quantize → ตารางว่าเลเยอร์ไหนถูก quantize จริง
2. `lab.evaluate(run)` คุณภาพเทียบ BF16 (PPL, top-1 agree) + คำตอบภาษาไทยเทียบกันแบบเคียงข้าง
3. `lab.save(run)` บันทึก + ใส่หัว MTP + push ขึ้น Hub + เพิ่มแถวในตารางเปรียบเทียบ + คืนหน่วยความจำ

**ปรับตาม grader (Participant Briefing):** คะแนน S = 100 · A · s_E^0.6 · s_T^0.2 · s_L^0.2 โดย A (ความแม่นยำเทียบ BF16) คูณทุกอย่าง ถ้าความแม่นยำต่ำกว่า 85% ของ baseline คะแนนแทบเป็นศูนย์ ดังนั้น notebook นี้จึง
1. **calibrate ด้วย trace ที่ MANGO สร้างเอง** (self-calibration, Williams et al. NAACL 2025, arXiv 2410.17170) ครบ 6 ประเภทของ grader: chat, RAG, tool, reasoning, ไทย, long-context โดย**เปิด thinking** เหมือนตอนแข่ง ทำให้ quantizer เห็น activation ของ `<think>` ยาว ๆ และ tool call จริง
2. วัดคุณภาพบน **trace ที่กันไว้ (held-out)** เป็นตัวชี้วัดหลัก (`dppl_trace_%`, `top1_trace_%`)
3. ทำ **W8A8 ก่อน** เพราะเสี่ยงต่อ A น้อยที่สุด แล้วค่อย W4A16

**ลำดับและงบเวลา A100** (ประมาณการ ยังไม่ได้จับเวลาบนโมเดลจริง ระหว่างรันจะเห็นความคืบหน้าทีละเลเยอร์)

| ส่วน | ทำครั้งเดียว / ต่อสูตร | เวลาโดยประมาณ |
|---|---|---|
| Setup + ดาวน์โหลด | ครั้งเดียว | 10–15 นาที |
| 0.7 self-calibration trace (ติดตั้ง vLLM + สร้าง trace) | ครั้งเดียว เก็บบน Drive | 20–35 นาที |
| calibration set + baseline | ครั้งเดียว | 10–15 นาที |
| สูตร 1 `sq-gptq-w8a8` (ตัวหลัก) | ต่อสูตร | 40–70 นาที |
| สูตร 2 `gptq-w4a16` | ต่อสูตร | 40–70 นาที |
| สูตร 3–5 | ทำถ้ามีเวลา | 40–70 นาทีต่อสูตร |
| ตารางเปรียบเทียบ + handoff | ครั้งเดียว | 2 นาที |

ถ้า A100 จำกัด ให้ทำแค่ Setup → สูตร 1 → สูตร 2 → handoff ก็ส่งงานได้แล้ว ถ้า session หลุด ผลที่บันทึกแล้วไม่หาย (อยู่บน Drive และ Hub) ให้รันใหม่ตั้งแต่ขั้น 0.2 แล้วข้ามสูตรที่ทำเสร็จแล้ว

Runtime: **GPU A100 + High-RAM** (ต้องใช้ RAM ≥ 40 GB เพื่อถือโมเดล BF16 บน CPU)

## 0.1 · ติดตั้ง (ครั้งเดียวต่อ runtime)
**[VM] ไม่ต้องรัน cell นี้ (comment ไว้แล้ว)** — ถ้ารันจะ pip install ทับ kernel และ `os.kill` ฆ่า kernel ทิ้ง venv ติดตั้งไว้แล้วที่ `~/nut/venvs/quant` ถ้าต้องสร้างใหม่ให้ดูคำสั่งใน cell ถัดไป

llm-compressor 0.12.0 ต้องใช้ transformers 5.9–5.10 ซึ่งใหม่กว่าที่ Colab ติดตั้งมา cell นี้จะติดตั้งแล้ว **ปิด session เอง** (เห็นข้อความว่า session crashed ได้ ไม่ต้องตกใจ) จากนั้นรันต่อที่ 0.2

ถ้ากด Runtime → Restart session ทีหลัง ไม่ต้องรันขั้นนี้ซ้ำ แต่ถ้า Disconnect and delete runtime ต้องรันใหม่

In [ ]:
# [VM] ไม่รัน: เฉพาะ Colab — บน VM ใช้ kernel "GreenMind quant (llmcompressor 0.12)" ที่ติดตั้งไว้แล้ว
# !pip install "llmcompressor==0.12.0" hf_transfer matplotlib
# import os; os.kill(os.getpid(), 9)   # รีสตาร์ท session ให้โหลดเวอร์ชันใหม่
#
# ถ้าต้องสร้าง venv ใหม่บน VM (รันใน terminal ไม่ใช่ใน notebook):
#   uv pip freeze --python ~/work/.venv-quant/bin/python > /tmp/quant_freeze.txt
#   uv venv ~/nut/venvs/quant --python 3.12 --seed
#   uv pip install --python ~/nut/venvs/quant/bin/python --extra-index-url https://download.pytorch.org/whl/cu132 \
#       --index-strategy unsafe-best-match -r /tmp/quant_freeze.txt ipykernel matplotlib hf_transfer
#   ~/nut/venvs/quant/bin/python -m ipykernel install --user --name greenmind-quant --display-name "GreenMind quant (llmcompressor 0.12)"

## 0.2 · Toolkit + path บน VM + ล็อกอิน Hugging Face
**[VM แก้แล้ว]** แทน Google Drive ด้วยโฟลเดอร์ใน `~/nut` และใช้ HF cache เดิม `~/work/hf`

`HF_TOKEN` (สิทธิ์ **write**) อ่านจาก environment หรือจาก `hf auth login` ที่ทำไว้ใน terminal ห้ามใส่ token ลงใน notebook ถ้าไม่มี token notebook ยังรันได้ แต่จะไม่ push ขึ้น Hub

✅ ควรเห็น: `ล็อกอิน Hugging Face เป็น <ชื่อของคุณ>` (หรือคำเตือนว่าไม่ push) และ `toolkit พร้อม`

In [ ]:
# [VM แก้แล้ว] ไม่มี google.colab / Drive — ใช้โฟลเดอร์บนดิสก์ VM แทน
# from google.colab import drive, userdata
# drive.mount('/content/drive')

import os, sys, shutil, zipfile, pathlib
HOME = pathlib.Path.home()
NUT = HOME / 'nut'
GM_DIR = NUT / 'gm'                                # toolkit (แตกจาก zip ใหม่ทุกครั้ง อย่าแก้โค้ดในนี้)
ZIP = NUT / 'greenmind-quant.zip'
for d in (GM_DIR, NUT / 'greenmind-quant'):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(NUT)
shutil.move(str(NUT / 'greenmind-quant'), str(GM_DIR))
sys.path.insert(0, str(GM_DIR))

os.environ.update(
    HF_HOME=str(HOME / 'work' / 'hf'),               # cache เดิมที่มี MANGO BF16 แล้ว (อย่าตั้งใหม่ ไม่งั้นโหลดซ้ำ 18 GB)
    HF_XET_HIGH_PERFORMANCE='1',                   # huggingface_hub 1.x เลิกใช้ hf_transfer แล้ว
    GM_RESULTS_DIR=str(NUT / 'results'),
    GM_MODELS_DIR=str(NUT / 'models'),             # checkpoint ที่ quantize แล้ว (ชุดละ 8–14 GB)
    GM_VENV_ROOT=str(NUT / 'venvs'),
    TOKENIZERS_PARALLELISM='false',
    PYTHONPATH=str(GM_DIR),                        # ให้ subprocess (เช่น gm.handoff, gm.calib_trace) หาโมดูลเจอ
)

from huggingface_hub import HfApi, get_token
if get_token():                                    # HF_TOKEN ใน env หรือจาก `hf auth login`
    HF_USER = HfApi().whoami()['name']
    print('ล็อกอิน Hugging Face เป็น', HF_USER)
else:
    HF_USER = None
    print('⚠️ ไม่พบ HF token → จะไม่ push ขึ้น Hub (รัน `hf auth login` ใน terminal แล้วรัน cell นี้ใหม่ถ้าต้องการ push)')
print('toolkit พร้อม:', sorted(os.listdir(GM_DIR / 'gm')))

## 0.3 · ตรวจเวอร์ชันและฮาร์ดแวร์
✅ ควรเห็น: llmcompressor 0.12.0, transformers 5.9.x หรือ 5.10.x, `CUDA ใช้ได้: True`, GPU เป็น A100 และ RAM ≥ 40 GiB

**[VM เพิ่ม]** ตรวจว่า GPU ไม่มี process อื่น และ RAM ที่**ว่างจริง** (MemAvailable) ≥ 40 GiB ถ้าไม่ผ่าน cell จะหยุด เพราะถ้ารันต่อ OOM killer อาจฆ่า kernel นี้หรือ kernel ของคนอื่นทิ้ง ให้ปิด kernel/vLLM ตัวอื่นก่อน (หรือตั้ง `GM_SKIP_PRECHECK=1` ถ้ารู้ตัวว่าทำอะไร)

❌ transformers ยังเป็น 4.x → ยังไม่ได้รีสตาร์ทหลัง 0.1 ให้กด Runtime → Restart session แล้วรัน 0.2 ใหม่
❌ `CUDA ใช้ได้: False` บน runtime ที่เป็น GPU → torch ที่ pip ติดตั้งมาไม่ตรงกับ driver ของ Colab ให้ส่งบรรทัด `torch CUDA` และ `driver` มาให้ผมดู

In [ ]:
import gc, json, os, sys, time, pathlib, subprocess
import torch, transformers, llmcompressor, compressed_tensors
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from gm import explore as ex
from gm.common import RESULTS_DIR, package_versions

print(package_versions('llmcompressor', 'compressed-tensors', 'transformers', 'torch', 'datasets'))
assert transformers.__version__.startswith('5.'), 'transformers ยังเป็นเวอร์ชันเก่า: รีสตาร์ท session แล้วรัน 0.2 ใหม่'
print('CUDA ใช้ได้:', torch.cuda.is_available(), '| torch CUDA', torch.version.cuda)
try:
    print('driver:', subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                                     '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print('driver: ไม่พบ nvidia-smi (ไม่ได้อยู่บน GPU runtime)')
meminfo = {l.split(':')[0]: int(l.split()[1]) for l in open('/proc/meminfo')}
ram, ram_free = meminfo['MemTotal'] / 2**20, meminfo['MemAvailable'] / 2**20
print(f'RAM: {ram:.0f} GiB · ว่างจริง {ram_free:.0f} GiB')

# [VM เพิ่ม] VM ใช้ร่วมกันทั้งทีม: ห้ามมีงานอื่นถือ GPU/RAM ตอน quantize
other_gpu = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory',
                            '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
other_gpu = [l for l in other_gpu.splitlines() if l and not l.startswith(f'{os.getpid()},')]
problems = []
if other_gpu:
    problems.append('มี process อื่นใช้ GPU อยู่:\n  ' + '\n  '.join(other_gpu))
if ram_free < 40:
    problems.append(f'RAM ว่างเพียง {ram_free:.0f} GiB (ต้องการ ≥ 40 GiB เพื่อถือ BF16 บน CPU)')
if problems and os.environ.get('GM_SKIP_PRECHECK') != '1':
    raise RuntimeError('ยังไม่พร้อม quantize — ปิด kernel/vLLM ตัวอื่นก่อน:\n' + '\n'.join(problems))
print('✅ GPU และ RAM ว่างพอ' if not problems else '⚠️ ข้าม precheck: ' + ' | '.join(problems))

## 0.4 · ค่าที่ใช้กับทุกสูตร (ไม่ต้องแก้)
- **384 samples × 4096 tokens**: ยาวกว่าที่ CMKL ใช้ (512 × 2048) เพราะคำตอบแบบ thinking ใน grader ยาวหลายพัน token ถ้า calibrate แค่ 2048 token quantizer จะไม่เคยเห็น activation ช่วงท้ายของการคิด รวมราว 1–1.5 ล้าน token ซึ่งเกิน d_in ของเลเยอร์ใหญ่สุด (12,288) หลายสิบเท่า GPTQ จึงไม่ overfit
- ในนั้น **320 ตัวอย่างเป็น self-calibration trace** (ขั้น 0.7) อีก 64 ตัวอย่างเป็นแชทสาธารณะไทย/อังกฤษ เพื่อกันไม่ให้แคบเกินไป
- ทุกสูตรใช้ calibration set ชุดเดียวกัน ผลจึงเทียบกันได้ตรง ๆ
- `PUSH = True`: push checkpoint ขึ้น Hub เป็น repo **private** ชื่อ `<HF_USER>/mango15-<สูตร>` เพื่อให้ serving ดึงไปใช้ได้

ถ้า A100 เหลือน้อยมากจริง ๆ ให้ลด `NUM_SAMPLES = 256` ซึ่งจะเร็วขึ้นราวหนึ่งในสาม แลกกับคุณภาพที่อาจลดลงเล็กน้อย

In [ ]:
SOURCE = 'CMKL/MANGO1.5-Qwen3.5-9B'
NUM_SAMPLES, SEQ_LEN = 384, 4096
TRACE_N, TRACE_HOLDOUT = 320, 24          # self-calibration traces (0.7) + held-out traces for evaluation
PUSH = True
HF_USER = globals().get('HF_USER')        # มาจาก 0.2
print(f'source={SOURCE} · calibration={NUM_SAMPLES}×{SEQ_LEN} · push={PUSH and bool(HF_USER)} as {HF_USER}')
print('ผลลัพธ์จะถูกเก็บที่', RESULTS_DIR)

# [VM เพิ่ม] checkpoint ชุดละ 8–14 GB อยู่ใน GM_MODELS_DIR; disk เต็มระหว่าง save จะทำให้ checkpoint เสีย
free_gb = shutil.disk_usage(os.environ['GM_MODELS_DIR'].rsplit('/', 1)[0]).free / 1e9
print(f'disk ว่าง {free_gb:.0f} GB → เก็บ checkpoint ได้ราว {int((free_gb - 5) // 14)} ชุด')

## 0.5 · ดาวน์โหลดโมเดล
**[VM]** ใช้ cache `~/work/hf` ซึ่งมี MANGO BF16 ครบแล้ว (revision `c0e66f0d`, 18 GB) cell นี้จะเสร็จในไม่กี่วินาที ไม่ต้องแก้

✅ ควรเห็น: progress bar ของไฟล์ `.safetensors` แล้วตามด้วยตารางไฟล์ รวมราว 19 GB และมีไฟล์ที่ชื่อมี `mtp` (หัว MTP)

In [ ]:
from huggingface_hub import snapshot_download
t = time.time()
SRC_DIR = SOURCE if pathlib.Path(SOURCE).exists() else snapshot_download(SOURCE)
files = sorted(pathlib.Path(SRC_DIR).glob('*.safetensors'))
display(pd.DataFrame([{'file': f.name, 'GB': round(f.stat().st_size / 1e9, 2)} for f in files]))
print(f'รวม {sum(f.stat().st_size for f in files) / 1e9:.1f} GB · {(time.time() - t) / 60:.1f} นาที')

## 0.6 · Tokenizer และ chat template
calibration data ต้องหน้าตาเหมือนแชทจริง cell นี้แสดงข้อความที่โมเดลเห็นจริงหลังผ่าน chat template

✅ ควรเห็น: system prompt ของ Mango, แท็ก `<|im_start|>user` และจำนวน token ของประโยคไทยเทียบกับอังกฤษ

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(SRC_DIR)
msgs = [{'role': 'user', 'content': 'สวัสดีครับ ช่วยสรุปข่าววันนี้ให้หน่อย'}]
print(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False))
th = 'การ quantize ทำให้โมเดลเล็กลงและสร้างคำตอบได้เร็วขึ้น'
en = 'Quantization makes the model smaller and lets it generate answers faster'
print(f"ไทย {len(tok(th)['input_ids'])} tokens · อังกฤษ {len(tok(en)['input_ids'])} tokens")

## 0.7a · Self-calibration: ให้ MANGO BF16 สร้าง trace แบบเดียวกับ grader (ครั้งเดียว เก็บบน Drive)
**หลักการ:** quantizer (GPTQ/SmoothQuant) ปรับ weight ให้ output ผิดน้อยที่สุด *บน activation ที่มันเห็น* ถ้า calibration ไม่เหมือนงานจริง มันจะไปรักษาส่วนที่ไม่สำคัญ งานของ Williams et al. (NAACL 2025) พบว่าข้อความที่โมเดลสร้างเองใช้ calibrate ได้ดีเท่าหรือดีกว่าข้อมูลจริง เพราะตรงกับ distribution ของโมเดลเอง

cell นี้
1. **[VM]** ใช้ vLLM venv ที่มีอยู่แล้ว `~/work/.venv` (vLLM 0.27.1) ไม่ต้องติดตั้งใหม่ — **ต้องปิด `vllm serve` ของทีมก่อน** เพราะ cell นี้จอง GPU 90% (log อยู่ที่ `~/nut/results/quantization/calib_trace.log`)
2. สร้าง prompt แบบเดียวกับ grader 6 ประเภท (chat, RAG, tool calling พร้อม tools, reasoning, ไทย, long-context) จากข้อมูลเปิด (UltraChat, Wikipedia shard แรก, GSM8K/MBPP, WangchanThaiInstruct) **ไม่มีข้อสอบของ grader** ตามกติกา
3. ให้ MANGO BF16 ตอบด้วย **thinking เปิด, temperature 0** แล้วเก็บ prompt + คำตอบเป็น calibration 320 ตัวอย่าง + กันไว้วัดผล 24 ตัวอย่าง
4. ทดสอบเพิ่ม 24 คำถามที่ max_tokens 8192 เพื่อดูว่า BF16 คิดยาวเท่าไรและโดนตัดที่ 8192 บ่อยไหม (ข้อมูลสำคัญให้ serving)

✅ ควรเห็น: log ติดตั้ง → `generated 344 traces in N min` → ตารางสถิติต่อประเภท (`gen_tokens` เฉลี่ย, `finish`) และ `bf16_full_length_probe`
❌ ถ้าล้มเหลว (เช่น ติดตั้ง vLLM ไม่ได้) cell จะบอกและ **ใช้ calibration สาธารณะแทนอัตโนมัติ** ยังรันต่อได้ แต่ส่งข้อความ error ให้ผมดู

In [ ]:
from gm.common import run_live
TRACE = RESULTS_DIR / 'quantization' / 'self_calib_trace.jsonl'
# [VM แก้แล้ว] ใช้ vLLM venv ที่มีอยู่แล้วบน VM (vLLM 0.27.1 + datasets) แทน /content/venvs/serve
SERVE_PY = str(pathlib.Path.home() / 'work' / '.venv' / 'bin' / 'python')
if TRACE.exists():
    print('ใช้ trace เดิม:', TRACE)
else:
    try:
        if not pathlib.Path(SERVE_PY).exists():
            SERVE_PY = os.environ['GM_VENV_ROOT'] + '/serve/bin/python'
            if not pathlib.Path(SERVE_PY).exists():
                run_live(['bash', str(GM_DIR / 'setup_envs.sh'), 'serve'])
        # vLLM จอง GPU 90% → ต้องไม่มี process อื่น (รวม vllm serve ของทีม) ถือ GPU อยู่
        used, total = map(float, subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total',
                          '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.split(','))
        gpu_free = 1 - used / total
        assert gpu_free > 0.92, f'GPU ว่างเพียง {gpu_free:.0%} — ปิด vLLM/kernel ตัวอื่นก่อน'
        rc = run_live([SERVE_PY, '-m', 'gm.calib_trace', '--model', SRC_DIR, '--out', str(TRACE),
                       '--n', str(TRACE_N), '--holdout', str(TRACE_HOLDOUT), '--seq-cap', str(SEQ_LEN)],
                      log_path=RESULTS_DIR / 'quantization' / 'calib_trace.log')
        if rc != 0 or not TRACE.exists():
            raise RuntimeError(f'calib_trace exit code {rc}')
    except Exception as e:
        print('⚠️ self-calibration ล้มเหลว:', e, '→ ใช้ calibration สาธารณะแทน')
        TRACE = None
if TRACE:
    st = json.loads(TRACE.with_suffix('.stats.json').read_text())
    print('calibration traces ต่อประเภท:'); display(pd.DataFrame(st['calibration']).T)
    if 'bf16_full_length_probe' in st:
        print('BF16 ที่ max_tokens 8192 (finished_% ต่ำ = คิดยาวจนโดนตัด → แจ้ง serving):')
        display(pd.DataFrame(st['bf16_full_length_probe']).T)

## 0.7b · ประกอบ calibration set (ใช้กับทุกสูตร)
trace 320 ตัวอย่างจาก 0.7a + แชทสาธารณะ 64 ตัวอย่าง (ไทย 35% จาก WangchanThaiInstruct, อังกฤษ 65% จาก UltraChat) ตัดที่ 4096 tokens trace ที่กันไว้ 24 ตัวอย่าง (`split = eval`) **ไม่ถูกใช้ calibrate** แต่ใช้วัดผลใน `lab.evaluate`

✅ ควรเห็น: `loaded 320 user calibration rows`, สถิติความยาวเฉลี่ยหลายพัน token และตัวอย่างที่มี `<think>` กับ `<tool_call>` อยู่ในข้อความ
❌ ถ้า `thai_samples_%` ใกล้ 0 แปลว่าโหลดข้อมูลไทยไม่ได้ (ดู warning ด้านบน) **อย่ารันต่อ** เพราะคุณภาพภาษาไทยจะเสีย

In [ ]:
from gm.calib_data import build_calibration_dataset
t = time.time()
calib_ds = build_calibration_dataset(tok, NUM_SAMPLES, SEQ_LEN, thai_ratio=0.35, seed=42,
                                     calib_jsonl=str(TRACE) if TRACE else None)
calib_stats = ex.describe_calibration(calib_ds, tok)
CALIB_TAG = f"{'self-trace' if TRACE else 'public'} {NUM_SAMPLES}x{SEQ_LEN}"
print(f'\n{CALIB_TAG} · {(time.time() - t) / 60:.1f} นาที')

## 0.8 · โหลดโมเดล BF16 และดูโครงสร้าง
โหลดบน CPU เพราะ llm-compressor จะย้ายเลเยอร์ขึ้น GPU ทีละชั้นตอน quantize

✅ ควรเห็น: `layer_types` เป็น linear_attention 24 กับ full_attention 8 และตารางกลุ่มเลเยอร์ที่ MLP มีพารามิเตอร์มากที่สุด คอลัมน์ `plan` บอกว่ากลุ่มไหนจะถูก quantize ส่วนที่คงเป็น BF16 (`lm_head`, embedding, vision) คือเหตุที่ INT4 ไม่เล็กลงถึง 4 เท่า

In [ ]:
from gm.lab import QuantLab
from gm.common import MODELS_DIR                  # [VM แก้แล้ว] default ของ QuantLab คือ /content/models
lab = QuantLab(SRC_DIR, SOURCE, tok, calib_ds, calib_stats, RESULTS_DIR, models_dir=MODELS_DIR,
               hf_user=HF_USER if PUSH else None, seq_len=SEQ_LEN,
               trace_path=TRACE)
inventory = lab.inventory()

## 0.9 · BF16 baseline (คำนวณครั้งแรกครั้งเดียว เก็บบน Drive)
วัดโมเดลต้นฉบับไว้เป็นตัวเทียบของทุกสูตร
- PPL และ top-1 บน **trace ที่กันไว้** (ตัวชี้วัดหลัก เพราะเหมือนงานใน grader ที่สุด) + PPL ไทยและอังกฤษ (Wikipedia shard สุดท้าย / WikiText)
- คำตอบตัวอย่าง 3 ข้อ
- activation จริงที่ไหลเข้า `layers.15.mlp.down_proj` เพื่อใช้ใน 0.10 และในการตรวจรายเลเยอร์ของแต่ละสูตร

✅ ควรเห็น: ตาราง PPL (ไทยมักสูงกว่าอังกฤษ เป็นเรื่องปกติ) และคำตอบภาษาไทยที่อ่านรู้เรื่อง

In [ ]:
base = lab.baseline()

## 0.10 · ห้องทดลอง: ปัดเลขตรง ๆ (RTN) แบบต่าง ๆ บนเลเยอร์จริง
ใช้เวลาไม่กี่วินาที และเป็นเส้นฐานให้รู้ว่า GPTQ / AWQ / SmoothQuant ช่วยได้เท่าไร
- **weight err**: weight เปลี่ยนไปเท่าไร
- **output err**: ผลคูณกับ activation จริงเปลี่ยนไปเท่าไร ซึ่งเลเยอร์ถัดไปรับรู้จริง **ตัวนี้สำคัญกว่า**

✅ ควรเห็นแนวโน้มนี้: W8 error น้อยมาก · W4 per-channel สูง · W4 g128 ลดลงชัดเจน (เหตุที่ทุกคนใช้ group 128) · g32 ดีขึ้นอีกแต่ไฟล์ใหญ่ขึ้น · W3 พุ่งขึ้น

In [ ]:
rtn = lab.rtn_lab()

## สูตร 1 · `sq-gptq-w8a8` — ตัวหลักสำหรับ grader
**SmoothQuant (α 0.8) แล้วตามด้วย GPTQ, INT8 weight + INT8 activation แบบ dynamic per-token** — ใน grader คะแนนทั้งหมดถูกคูณด้วย A (ความแม่นยำเทียบ BF16) ถ้า A ตก ทุกอย่างพัง งานวิจัย *Quantization Hurts Reasoning?* (arXiv 2504.04823) พบว่า W8A8 แทบไม่ทำให้ reasoning model เสีย ขณะที่ 4-bit เริ่มเสียกับโจทย์ยาก และ W8A8 ใช้ INT8 tensor core ของ A100 จึงช่วย prefill ของ RAG / long-context (TTFT) ตอน peak 4 req/s

**cell 1 quantize:** จะเห็น log `Quantizing model.language_model.layers.N…` ไล่ N จาก 0 ถึง 31 (ใช้ดูความคืบหน้า) และ `GPTQ | METRIC - error` ต่อเลเยอร์ ถ้าเป็น `nan` หรือ `inf` ให้หยุด จบแล้วจะเห็นตารางว่าเลเยอร์ไหนถูก quantize จริง และการเทียบกับการปัดตรง ๆ บนเลเยอร์ 15
**cell 2 evaluate:** ตาราง PPL / Δ PPL % / top-1 agree เทียบ BF16 และคำตอบเคียงข้างกัน
**cell 3 save:** บันทึก, ใส่หัว MTP, push และเพิ่มแถวในตารางเปรียบเทียบ

✅ ควรเห็น: Δ PPL-trace ต่ำมาก (ใกล้ 0) · top-1 trace ≥ 97% · quant summary ต้องมี activations เป็น `INT8 dynamic` · ขนาดราว 11–12 GB

In [ ]:
run1 = lab.quantize('sq-gptq-w8a8', notes=f'sq-gptq-w8a8 · {CALIB_TAG}')

In [ ]:
lab.evaluate(run1)

In [ ]:
lab.save(run1, push=PUSH)

## สูตร 2 · `gptq-w4a16` — ตัวเลือกประหยัดพลังงานสูงสุด
**GPTQ, INT4 weight, group 128, symmetric** — weight เล็กที่สุดที่ Marlin รองรับ decode (memory-bound) จึงเร็วสุดและเหลือหน่วยความจำให้ KV cache มากสุด แต่เสี่ยงกับ reasoning ยาว ๆ ใช้เมื่อคุณภาพผ่านเกณฑ์เท่านั้น

**cell 1 quantize:** จะเห็น log `Quantizing model.language_model.layers.N…` ไล่ N จาก 0 ถึง 31 (ใช้ดูความคืบหน้า) และ `GPTQ | METRIC - error` ต่อเลเยอร์ ถ้าเป็น `nan` หรือ `inf` ให้หยุด จบแล้วจะเห็นตารางว่าเลเยอร์ไหนถูก quantize จริง และการเทียบกับการปัดตรง ๆ บนเลเยอร์ 15
**cell 2 evaluate:** ตาราง PPL / Δ PPL % / top-1 agree เทียบ BF16 และคำตอบเคียงข้างกัน
**cell 3 save:** บันทึก, ใส่หัว MTP, push และเพิ่มแถวในตารางเปรียบเทียบ

✅ ควรเห็น: Δ PPL-trace ไม่เกินราว 3–5% · top-1 trace ≥ 93% · คำตอบภาษาไทยยังอ่านรู้เรื่อง · ขนาดราว 8 GB · MTP 15 tensors

In [ ]:
run2 = lab.quantize('gptq-w4a16', notes=f'gptq-w4a16 · {CALIB_TAG}')

In [ ]:
lab.evaluate(run2)

In [ ]:
lab.save(run2, push=PUSH)

## สูตร 3 · `mixed-w4w8` — ประนีประนอม
**GPTQ แบบ mixed precision: MLP เป็น INT4 g128 ส่วน attention และ Gated DeltaNet เป็น INT8** — รักษาเส้นทาง recurrent state ไว้ที่ 8-bit แต่ยังเล็กกว่า W8 ทั้งตัว

**cell 1 quantize:** จะเห็น log `Quantizing model.language_model.layers.N…` ไล่ N จาก 0 ถึง 31 (ใช้ดูความคืบหน้า) และ `GPTQ | METRIC - error` ต่อเลเยอร์ ถ้าเป็น `nan` หรือ `inf` ให้หยุด จบแล้วจะเห็นตารางว่าเลเยอร์ไหนถูก quantize จริง และการเทียบกับการปัดตรง ๆ บนเลเยอร์ 15
**cell 2 evaluate:** ตาราง PPL / Δ PPL % / top-1 agree เทียบ BF16 และคำตอบเคียงข้างกัน
**cell 3 save:** บันทึก, ใส่หัว MTP, push และเพิ่มแถวในตารางเปรียบเทียบ

✅ ควรเห็น: คุณภาพอยู่ระหว่าง gptq-w4a16 กับ W8 · ตาราง quant summary ต้องมีทั้ง INT4 และ INT8

In [ ]:
# [VM ไม่รัน] disk เหลือ ~45 GB พอแค่สูตร 1–2 — จะรันได้เมื่อ push สูตรก่อนหน้าแล้วลบ ~/nut/models/mango15-<สูตร> ที่ไม่ใช้
# run3 = lab.quantize('mixed-w4w8', notes=f'mixed-w4w8 · {CALIB_TAG}')

In [ ]:
# [VM ไม่รัน] ตามสูตร 3 ด้านบน
# lab.evaluate(run3)

In [ ]:
# [VM ไม่รัน] ตามสูตร 3 ด้านบน
# lab.save(run3, push=PUSH)

## สูตร 4 · `gptq-w8a16` — ตัวสำรองด้านคุณภาพ
**GPTQ, INT8 weight-only, per-channel** — คุณภาพใกล้ BF16 มากที่สุด ใช้ถ้า W8A8 มีปัญหาตอนโหลดใน vLLM หรือคุณภาพไม่ผ่าน

**cell 1 quantize:** จะเห็น log `Quantizing model.language_model.layers.N…` ไล่ N จาก 0 ถึง 31 (ใช้ดูความคืบหน้า) และ `GPTQ | METRIC - error` ต่อเลเยอร์ ถ้าเป็น `nan` หรือ `inf` ให้หยุด จบแล้วจะเห็นตารางว่าเลเยอร์ไหนถูก quantize จริง และการเทียบกับการปัดตรง ๆ บนเลเยอร์ 15
**cell 2 evaluate:** ตาราง PPL / Δ PPL % / top-1 agree เทียบ BF16 และคำตอบเคียงข้างกัน
**cell 3 save:** บันทึก, ใส่หัว MTP, push และเพิ่มแถวในตารางเปรียบเทียบ

✅ ควรเห็น: Δ PPL ใกล้ 0 · top-1 agree สูงที่สุดในทุกสูตร

In [ ]:
# [VM ไม่รัน] ตัวสำรอง ใช้เมื่อ W8A8 โหลดใน vLLM ไม่ได้/คุณภาพไม่ผ่าน — ก่อนรันให้ลบ checkpoint ที่ไม่ใช้ใน ~/nut/models (ชุดนี้ ~14 GB)
# run4 = lab.quantize('gptq-w8a16', notes=f'gptq-w8a16 · {CALIB_TAG}')

In [ ]:
# [VM ไม่รัน] ตามสูตร 4 ด้านบน
# lab.evaluate(run4)

In [ ]:
# [VM ไม่รัน] ตามสูตร 4 ด้านบน
# lab.save(run4, push=PUSH)

## สูตร 5 · `awq-w4a16` — ทางเลือก ถ้ามีเวลา
**AWQ, INT4 asymmetric, group 128** — สูตรเดียวกับรุ่นทางการของ CMKL ต่างกันที่ calibration (ของเราเป็น self-calibration แบบ trace ของ grader) ถ้าเวลาไม่พอให้ข้าม เพราะรุ่นของ CMKL (`CMKL/MANGO1.5-Qwen3.5-9B-AWQ-W4A16-mtp`) ใช้แทนได้

**cell 1 quantize:** จะเห็น log `Quantizing model.language_model.layers.N…` ไล่ N จาก 0 ถึง 31 (ใช้ดูความคืบหน้า) และ `GPTQ | METRIC - error` ต่อเลเยอร์ ถ้าเป็น `nan` หรือ `inf` ให้หยุด จบแล้วจะเห็นตารางว่าเลเยอร์ไหนถูก quantize จริง และการเทียบกับการปัดตรง ๆ บนเลเยอร์ 15
**cell 2 evaluate:** ตาราง PPL / Δ PPL % / top-1 agree เทียบ BF16 และคำตอบเคียงข้างกัน
**cell 3 save:** บันทึก, ใส่หัว MTP, push และเพิ่มแถวในตารางเปรียบเทียบ

✅ ควรเห็น: ขั้นตรวจรายเลเยอร์จะถูกข้าม (ปกติสำหรับ AWQ) ให้ดูตารางคุณภาพแทน

In [ ]:
# [VM ไม่รัน] ซ้ำกับ CMKL/MANGO1.5-Qwen3.5-9B-AWQ-W4A16-mtp และ disk/เวลาไม่พอ — รันเมื่อเหลือเวลาและลบ checkpoint เก่าแล้ว
# run5 = lab.quantize('awq-w4a16', notes=f'awq-w4a16 · {CALIB_TAG}')

In [ ]:
# [VM ไม่รัน] ตามสูตร 5 ด้านบน
# lab.evaluate(run5)

In [ ]:
# [VM ไม่รัน] ตามสูตร 5 ด้านบน
# lab.save(run5, push=PUSH)

## สรุป · ตารางเปรียบเทียบทุกสูตร
ทุกครั้งที่ `lab.save` ทำงานจะเพิ่ม 1 แถวใน `compare.csv` บน Drive ตารางนี้ตอบคำถามว่า **"quantize ด้วยอะไร ได้ผลแบบไหน"**

กราฟ: แกน X คือขนาดไฟล์ (เล็ก = decode เร็ว) แกน Y คือ PPL ไทยที่เพิ่มขึ้น (ต่ำ = คุณภาพดี) จุดที่อยู่ **มุมซ้ายล่าง** ดีที่สุด

In [ ]:
cmp = pd.read_csv(RESULTS_DIR / 'quantization' / 'compare.csv')
latest = cmp.groupby('variant').tail(1).reset_index(drop=True)
cols = ['variant', 'size_gb', 'mtp', 'dppl_trace_%', 'top1_trace_%', 'dppl_th_%', 'top1_th_%',
        'dppl_en_%', 'top1_en_%', 'probe_rtn_err_%', 'probe_out_err_%', 'quant_min', 'hub']
display(latest[[c for c in cols if c in latest.columns]])
fig = ex.plot_compare(cmp); plt.show()

## ส่งมอบให้ serving
สร้าง `results/handoff/HANDOFF.md` ซึ่งมีลำดับสูตรที่ควร benchmark (W8A8 ก่อน), ตารางคุณภาพ (รวม trace), ชื่อ repo, คำสั่ง `vllm serve` ที่ตรงสัญญาของ grader (tool parser `qwen3_coder`, usage ใน stream, max-model-len 49152) ทั้งแบบปิดและเปิด MTP และ checklist ที่ต้อง A/B บน Validation พร้อม push model card เป็น README ของแต่ละ repo

serving ใช้ `serve/serve_grader.sh` เปิด server และ `python -m gm.loadtest` (หรือ `serve/sweep.sh`) จำลอง grader บน VM เพื่อเลือก MTP / max-num-seqs / clock ก่อนส่ง Validation

เกณฑ์เริ่มต้น: PPL เพิ่มไม่เกิน 5% และต้องมีหัว MTP (ปรับได้เมื่อกรรมการประกาศเกณฑ์จริง)

✅ ควรเห็น: รายการ "Benchmark these, in this order" และตารางคุณภาพ ช่อง `loads in vLLM` จะเป็น `not yet` ซึ่งปกติ เพราะ serving จะเป็นคนโหลดใน vLLM ก่อน

**สิ่งที่ต้องทำต่อ:** (1) ส่งลิงก์ `HANDOFF.md` บน Drive ให้เพื่อนที่ทำ serving (2) เพิ่มเพื่อนเป็นสมาชิกของ repo private บน Hugging Face หรือย้าย repo ไปไว้ใน organization ของทีม

In [ ]:
cmd = [sys.executable, '-m', 'gm.handoff', '--results-dir', str(RESULTS_DIR), '--max-ppl-increase', '5']
if PUSH and HF_USER:
    cmd.append('--push-cards')
run_live(cmd)
display(Markdown((RESULTS_DIR / 'handoff' / 'HANDOFF.md').read_text()))